# 🎬 MiniMax H3 Ref2VA — 4 / 8 / 20 ステップ切替・音声長音声末尾カット修正版 v5

**目的:** ルカの画像と元のナレーションから、音声に合わせて動く短い動画を試し、InfiniteTalkと**速度・口パク・画質**を比較します。

- 画像と音声を**最初にアップロード**、その後に環境・モデルを用意します。
- **MiniMax H3 Ref2VA**（画像/音声参照）＋ **4ステップTurbo（成功済み）・8ステップTurbo（実験） LoRA**を使用。ComfyUIの**標準ノード**からローカルAPIで生成します。
- **A100 40GB / 80GB・G4**を想定。40GBではCPUへのオフロードによって遅くなったり、RAM不足で失敗する可能性があります。
- 生成後は**生成音声付きMP4 / 元音声を重ねた比較MP4 / 時間レポート**をZIPで取得します。

> **重要:** Ref2VAの参照音声は、話し方・声・動きの参考です。**渡した日本語音声を1語一句そのまま、完全に同じタイミングで発話する保証はありません。** 元音声を映像に重ねた版も口パクがずれる可能性があるため、必ず視聴して比較してください。
>
> 公式対応は1クリップ最大15秒程度。初期設定は**音声全長（モデル上限約14.38秒）・8ステップ品質検証**です。短い試用には②で秒数を5にしてください。約40GB以上のモデル取得と十分なシステムRAM・ディスクが必要です。モデルはColabセッションが消えると再ダウンロードになります。

**v2の改善:** ①では画像と音声を同時にアップロードできます。すでにアップロード済みでエラーになった場合は①の `REUSE_PREVIOUS_UPLOADS=True` を選択してください。

**実行順:** ①入力 → ②設定 → ③準備 → ④モデル → ⑤起動 → ⑥生成 → ⑦取得。初回以降は同じランタイムなら①②⑥⑦だけ繰り返せます。

参考: [MiniMax公式](https://github.com/MiniMax-AI/MiniMax-H3) / [ComfyUI公式Ref2VA](https://docs.comfy.org/tutorials/video/minimax/minimax-h3-native) / [Comfy-Org配布モデル](https://huggingface.co/Comfy-Org/MiniMax-H3)
> **v3更新:** ②で4/8/20ステップを選べます。8ステップは専用のRef2VA LoRA（約2GB）を④で別途取得し、推奨の動画/音声シフト6/3を使用します。15秒の音声改善は保証しません。

**v4修正:** v3で初期値が5秒・4ステップだったため、15秒音声が5秒に切られていました。v4は音声長を自動検出（上限約14.38秒）、8ステップを初期選択。⑥の実行前に秒数・ステップ・音声切り詰めを明示します。すでにColabにモデルがある場合はモデル本体を再取得する必要はありません（8ステップLoRAは追加取得）。


In [ ]:
#@title ① ルカ画像＋音声をアップロード（同時選択対応・再アップロード不要モードあり）
#@markdown 前のv1で両方アップロードした直後にエラーが出た場合は **True** にして実行。
REUSE_PREVIOUS_UPLOADS = False #@param {type:"boolean"}
from pathlib import Path
from google.colab import files
from IPython.display import display, Audio, Image
import shutil

WORK = Path('/content/h3_ruka_test')
UP = WORK / 'inputs'
UP.mkdir(parents=True, exist_ok=True)
IMAGE_EXT = {'.png', '.jpg', '.jpeg', '.webp'}
AUDIO_EXT = {'.wav', '.mp3', '.m4a', '.ogg', '.flac', '.aac'}

if REUSE_PREVIOUS_UPLOADS:
    # files.upload() は Colab の現在の作業フォルダに保存する。
    # 直前の画像と音声をそれぞれ別々に選ぶ（別形式の古いファイルは選ばない）。
    candidates = [p for p in Path.cwd().iterdir()
                  if p.is_file() and p.stat().st_size > 0 and
                  p.suffix.lower() in (IMAGE_EXT | AUDIO_EXT)]
    imgs = sorted((p for p in candidates if p.suffix.lower() in IMAGE_EXT),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    auds = sorted((p for p in candidates if p.suffix.lower() in AUDIO_EXT),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    assert imgs and auds, ('前にアップロードした画像または音声が見つかりません。'
                          'REUSE_PREVIOUS_UPLOADS=False で再アップロードしてください。')
    src_img, src_audio = imgs[0], auds[0]
    print('♻ Colabに残っているファイルを再利用')
    print('画像:', src_img, '／ 音声:', src_audio)
    image_name, image_data = src_img.name, src_img.read_bytes()
    audio_name, audio_data = src_audio.name, src_audio.read_bytes()
else:
    print('📁 画像1枚＋音声1個を同時に選んでアップロードしてください。')
    chosen = dict(files.upload())
    assert chosen, '画像と音声をアップロードしてください。'
    def _separate(data):
        imgs = [(name, content) for name, content in data.items()
                if Path(name).suffix.lower() in IMAGE_EXT]
        auds = [(name, content) for name, content in data.items()
                if Path(name).suffix.lower() in AUDIO_EXT]
        return imgs, auds
    imgs, auds = _separate(chosen)
    if not imgs:
        print('画像が未選択なので、画像だけ追加してください。')
        chosen.update(files.upload())
        imgs, auds = _separate(chosen)
    if not auds:
        print('音声が未選択なので、音声だけ追加してください。')
        chosen.update(files.upload())
        imgs, auds = _separate(chosen)
    assert len(imgs) == 1, f'画像は1枚を選んでください（現在{len(imgs)}枚）'
    assert len(auds) == 1, f'音声は1個を選んでください（現在{len(auds)}個）'
    image_name, image_data = imgs[0]
    audio_name, audio_data = auds[0]

# 同名ファイルがあるとき Colab が付ける「 (1)」付きファイル名でもOK。
# 後続セルで使用するパスを一定にする。
IMAGE_PATH = UP / ('character' + Path(image_name).suffix.lower())
AUDIO_PATH = UP / ('narration' + Path(audio_name).suffix.lower())
IMAGE_PATH.write_bytes(image_data)
AUDIO_PATH.write_bytes(audio_data)
assert IMAGE_PATH.stat().st_size > 0 and AUDIO_PATH.stat().st_size > 0
print('✅ 画像:', IMAGE_PATH, f'({IMAGE_PATH.stat().st_size:,} bytes)')
print('✅ 音声:', AUDIO_PATH, f'({AUDIO_PATH.stat().st_size:,} bytes)')
try:
    display(Image(filename=str(IMAGE_PATH), width=320))
except Exception as e:
    print('画像プレビュー省略:', e)
try:
    display(Audio(filename=str(AUDIO_PATH)))
except Exception as e:
    print('音声プレビュー省略:', e)
print('✅ 準備完了。②の生成設定に進んでください。')

In [ ]:
#@title ② 生成設定（デフォルト音声全長・8ステップ）
#@markdown **0なら音声を自動判定**します（生成映像は最大約14.38秒。元音声は全長保持し、足りない映像は最終フレームを延長）。5などを指定すると先頭だけ使います。
TEST_SECONDS = 0.0 #@param {type:"number"}
#@markdown 横長（解説動画）または縦長（ショート素材）
CANVAS = '横長 832x480' #@param ['横長 832x480','縦長 480x832','横長 960x544']
#@markdown **4ステップ** は実測済み。**8ステップ** は専用Ref2VA LoRAを追加ダウンロードして音声・画質の改善を検証する実験モード。
STEP_MODE = '8ステップ（音声品質を検証）' #@param ['4ステップ（実測済み・高速）','8ステップ（音声品質を検証）','20ステップ（LoRAなし・低速）']
NUM_STEPS = int(STEP_MODE.split('ステップ')[0])
USE_TURBO = NUM_STEPS in (4,8)
TURBO_4_STEP = NUM_STEPS == 4
SEED = 42 #@param {type:"integer"}
#@markdown 画面内のキャラクターに求める動き（日本語でも可。英語推奨）
MOTION_PROMPT = 'The same anime woman from <Picture 1> speaks naturally to the camera. Use <Audio 1> as a reference for her speech and timing. Natural lip movement, subtle head gestures and expressive hand movements. Keep her identity, hairstyle and clothing consistent. One person, steady camera, clean indoor studio, no subtitles.' #@param {type:"string"}
WIDTH,HEIGHT = {'横長 832x480':(832,480),'縦長 480x832':(480,832),'横長 960x544':(960,544)}[CANVAS]
assert TEST_SECONDS == 0 or 2 <= TEST_SECONDS <= 15, '音声テストは2～15秒、全長自動は0を指定してください'
assert WIDTH%32==0 and HEIGHT%32==0
print(f'✅ 設定確定: {WIDTH}x{HEIGHT} / {NUM_STEPS}ステップ / テスト秒数={TEST_SECONDS if TEST_SECONDS else "音声に自動追従（生成映像は上限約14.38秒・音声は全長）"}')

In [ ]:
#@title ③ GPU確認とComfyUIの環境構築（初回だけ）
import os, subprocess, sys, time, json
from pathlib import Path
COMFY = Path('/content/ComfyUI')

def run(cmd, cwd=None):
    print('▶', ' '.join(map(str,cmd)), flush=True)
    subprocess.run(list(map(str,cmd)),cwd=str(cwd) if cwd else None,check=True)

print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True))
import torch
assert torch.cuda.is_available(), 'GPUランタイムに接続してください'
print('torch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))
import shutil
disk_free_gib=shutil.disk_usage('/content').free/1024**3
print(f'空きディスク: {disk_free_gib:.1f} GiB')
assert disk_free_gib >= 45, 'モデル取得には約45GiB以上の空きが必要です。不要ファイルを削除するかディスク容量を増やしてください。'
if not (COMFY/'main.py').exists():
    run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git',COMFY])
# ComfyUI最新のPython依存関係を導入。大容量torchを入れ直さないよう基本は既存のCUDA対応torchを利用。
run([sys.executable,'-m','pip','install','-q','-r',COMFY/'requirements.txt'])
run([sys.executable,'-m','pip','install','-q','-U','huggingface_hub[hf_xet]','requests','pillow'])
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','-qq','ffmpeg'])
print('✅ ComfyUIのセットアップ完了。必要なモデルは次のセルで取得します')

In [ ]:
#@title ④ Ref2VAの量子化モデルとTurbo LoRAを取得（初回は約40GB）
#@markdown 通常はこのセルが最もダウンロード時間を使います。中断後の再実行ではHugging Faceが取得済みを再利用します。
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil, time
REPO = 'Comfy-Org/MiniMax-H3'
MODEL_FILES = [
  'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors',
  'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
  'vae/minimax_h3_video_vae_int8_convrot.safetensors',
  'vae/minimax_h3_audio_vae_fp32.safetensors',
]
LORA_NAME = (
    'minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors' if NUM_STEPS == 4 else
    'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors' if NUM_STEPS == 8 else None
)
# 4-stepは前回と同じモデルを使う。8-stepは別の専用LoRAが必要。
if NUM_STEPS == 4:
  MODEL_FILES.append('loras/'+LORA_NAME)
start_dl=time.perf_counter()
for filename in MODEL_FILES:
    dest=COMFY/'models'/filename
    dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.exists() and dest.stat().st_size>100*1024*1024:
        print('✅ 再利用:', filename, f'{dest.stat().st_size/2**30:.2f}GiB')
        continue
    print('⬇ ダウンロード:',filename,flush=True)
    hf_hub_download(repo_id=REPO,filename=filename,local_dir=str(COMFY/'models'))
    assert dest.exists() and dest.stat().st_size>100*1024*1024, f'モデルが不完全: {filename}'
    print('✅',f'{dest.stat().st_size/2**30:.2f}GiB',flush=True)
if NUM_STEPS == 8:
    target = COMFY / 'models' / 'loras' / LORA_NAME
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 100*1024*1024:
        print('✅ 8ステップRef2VA LoRAを再利用:', target.name)
    else:
        print('⬇ 8ステップ専用LoRAを取得（約2GB）:', LORA_NAME, flush=True)
        hf_hub_download(repo_id='lightx2v/Minimax-h3-Turbo', filename=LORA_NAME,
                        local_dir=str(COMFY/'models'/'loras'))
        assert target.exists() and target.stat().st_size > 100*1024*1024, '8ステップLoRA取得に失敗'
print(f'モデル準備時間：{(time.perf_counter()-start_dl)/60:.1f}分')

In [ ]:
#@title ⑤ ComfyUIをバックグラウンド起動・必要ノードを確認
import subprocess, os, json, time, requests
LOG = WORK/'comfyui.log'
URL='http://127.0.0.1:8188'
try:
    r=requests.get(URL+'/system_stats',timeout=2)
    online=r.ok
except Exception:
    online=False
if not online:
    cmd=[sys.executable,str(COMFY/'main.py'),'--listen','127.0.0.1','--port','8188','--disable-auto-launch','--disable-pinned-memory']
    # 48GB未満のGPUではモデルをシステムRAMへオフロードしてOOMリスクを軽減
    if torch.cuda.get_device_properties(0).total_memory / 1024**3 < 48:
        cmd.append('--lowvram')
        print('⚠ 48GB未満のGPUなので --lowvram で起動します')
    env=os.environ.copy()
    env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
    log_handle=open(LOG,'a',encoding='utf-8')
    COMFY_PROCESS=subprocess.Popen(cmd,cwd=str(COMFY),stdout=log_handle,stderr=subprocess.STDOUT,env=env,start_new_session=True)
    print('ComfyUI起動中…')
    for i in range(180):
        if COMFY_PROCESS.poll() is not None:
            raise RuntimeError('ComfyUIが終了しました。'+str(LOG)+' の末尾を確認してください')
        try:
            if requests.get(URL+'/system_stats',timeout=3).ok:
                online=True
                break
        except Exception: pass
        time.sleep(2)
assert online,'ComfyUI起動に時間がかかっています。comfyui.logをご確認ください'
objects=requests.get(URL+'/object_info',timeout=60).json()
required=['UNETLoader','CLIPLoader','VAELoader','MiniMaxH3ReferenceToVideo',
          'LoadImage','LoadAudio','BasicGuider','KSamplerSelect','BasicScheduler',
          'RandomNoise','SamplerCustomAdvanced','VAEDecode','VAEDecodeAudio',
          'CreateVideo','SaveVideo']
if USE_TURBO: required.append('LoraLoaderModelOnly')
if NUM_STEPS == 8: required.append('MiniMaxH3SigmaShift')
missing=[x for x in required if x not in objects]
assert not missing,'ComfyUIに必要なノードがありません: '+', '.join(missing)+'。ComfyUIの更新が必要です。'
print('✅ ComfyUI起動・必要ノードの存在確認が成功')
print('場所:',URL,'\nログ:',LOG)

In [ ]:
#@title ⑥ 画像＋音声から動画生成（音声長を確認してから実行）
import os, json, math, time, shutil, subprocess, requests, uuid
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, HTML
assert IMAGE_PATH.exists() and AUDIO_PATH.exists(), '先に①画像・音声をアップロード'
assert all((COMFY/'models'/x).exists() for x in MODEL_FILES), '先に④モデル取得を実行'
assert requests.get(URL+'/system_stats',timeout=15).ok, '先に⑤ComfyUIを起動'

# 毎回一意の実行フォルダを作り、前の出力を上書きしない
JOB = WORK/'outputs'/datetime.now().strftime('%Y%m%d_%H%M%S')
JOB.mkdir(parents=True,exist_ok=True)

def cmd_out(cmd):
    return subprocess.check_output(cmd,text=True).strip()

audio_duration=float(json.loads(cmd_out(['ffprobe','-v','error','-show_entries','format=duration','-of','json',str(AUDIO_PATH)]))['format']['duration'])
assert audio_duration>=2, '音声は2秒以上にしてください'
requested_sec=min(audio_duration, TEST_SECONDS if TEST_SECONDS>0 else audio_duration, 15.0)
# 24 fps、H3に推奨されるフレーム長17k+5に合わせる
k=max(3, min(20, round((requested_sec*24-5)/17)))
FRAMES=17*k+5
CLIP_SEC=FRAMES/24
print('\n'+'='*60,flush=True)
print(f'🎙 元音声: {audio_duration:.2f}秒',flush=True)
print(f'🎬 実際の生成動画: {CLIP_SEC:.2f}秒 ({FRAMES}フレーム)',flush=True)
print(f'⚙ ステップ: {NUM_STEPS} / LoRA: {LORA_NAME or "なし"}',flush=True)
if TEST_SECONDS > 0 and audio_duration-TEST_SECONDS > 0.2:
    print(f'⚠ テスト設定 {TEST_SECONDS:g}秒により、元音声の後半は使用されません。全長なら②で TEST_SECONDS=0 にしてください。',flush=True)
if requested_sec-CLIP_SEC>0.1:
    print(f'⚠ 生成映像は音声より {requested_sec-CLIP_SEC:.2f}秒短いです。⑦の元音声比較版では映像の末尾を静止延長し、音声を全長保存します。',flush=True)
print('='*60+'\n',flush=True)
if USE_TURBO:
    lora_file=COMFY/'models'/'loras'/LORA_NAME
    assert lora_file.exists() and lora_file.stat().st_size > 100*1024*1024, f'{NUM_STEPS}ステップ用LoRAがありません。④を実行してください: {lora_file}'
ref_seconds=min(audio_duration,CLIP_SEC)


# 参照音声をwavに統一（ステレオ32kHz）して前半だけ使用
ref_audio=JOB/'reference_audio.wav'
subprocess.run(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(AUDIO_PATH),'-t',str(ref_seconds),'-ar','32000','-ac','2',str(ref_audio)],check=True)

# ComfyUI/inputにコピーして参照する（外部URLや公開ポート不要）
inp=COMFY/'input'; inp.mkdir(parents=True,exist_ok=True)
staged_img=inp/f'h3_{uuid.uuid4().hex[:9]}{IMAGE_PATH.suffix}'
staged_audio=inp/f'h3_{uuid.uuid4().hex[:9]}.wav'
shutil.copy2(IMAGE_PATH,staged_img)
shutil.copy2(ref_audio,staged_audio)

model='minimax_h3_ref2va_pruned_int8_convrot.safetensors'
textenc='qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
video_vae='minimax_h3_video_vae_int8_convrot.safetensors'
audio_vae='minimax_h3_audio_vae_fp32.safetensors'

p={
'6': {'class_type':'UNETLoader','inputs':{'unet_name':model,'weight_dtype':'default'}},
'13':{'class_type':'CLIPLoader','inputs':{'clip_name':textenc,'type':'minimax','device':'default'}},
'11':{'class_type':'VAELoader','inputs':{'vae_name':video_vae}},
'24':{'class_type':'VAELoader','inputs':{'vae_name':audio_vae}},
'30':{'class_type':'LoadImage','inputs':{'image':staged_img.name,'upload':'image'}},
'60':{'class_type':'LoadAudio','inputs':{'audio':staged_audio.name}},
'104':{'class_type':'MiniMaxH3ReferenceToVideo','inputs':{
      'clip':['13',0], 'vae':['11',0], 'audio_vae':['24',0],
      'prompt':MOTION_PROMPT,'width':WIDTH,'height':HEIGHT,'length':FRAMES,'ref_image_size':'match',
      'ref_images.ref_image_0':['30',0], 'ref_audios.ref_audio_0':['60',0]}},
'17':{'class_type':'KSamplerSelect','inputs':{'sampler_name':'euler' if NUM_STEPS == 8 else 'res_multistep'}},
'15':{'class_type':'RandomNoise','inputs':{'noise_seed':SEED}},
'10':{'class_type':'VAEDecode','inputs':{'samples':['14',0],'vae':['11',0]}},
'23':{'class_type':'VAEDecodeAudio','inputs':{'samples':['14',0],'vae':['24',0]}},
'91':{'class_type':'CreateVideo','inputs':{'images':['10',0],'audio':['23',0],'fps':24}},
'92':{'class_type':'SaveVideo','inputs':{'video':['91',0],
        'filename_prefix':'H3_Ruka_'+JOB.name,'format':'auto','codec':'auto'}}
}
if USE_TURBO:
    p['7']={'class_type':'LoraLoaderModelOnly','inputs':{'model':['6',0],'lora_name':LORA_NAME,'strength_model':1.0}}
model_ref=['7',0] if USE_TURBO else ['6',0]
# 8-step 768p LoRA推奨の動画/音声シフト（6/3）。4-stepは実測済み構成のまま。
if NUM_STEPS == 8:
    p['8']={'class_type':'MiniMaxH3SigmaShift','inputs':{
        'model':model_ref,'shift_video':6.0,'shift_audio':3.0}}
    model_ref=['8',0]
p.update({
'16':{'class_type':'BasicGuider','inputs':{'model':model_ref,'conditioning':['104',0]}},
'9':{'class_type':'BasicScheduler','inputs':{'model':model_ref,'scheduler':'simple','steps':NUM_STEPS,'denoise':1.0}},
'14':{'class_type':'SamplerCustomAdvanced','inputs':{'noise':['15',0],'guider':['16',0],
       'sampler':['17',0],'sigmas':['9',0],'latent_image':['104',1]}}
})
(JOB/'workflow_api.json').write_text(json.dumps(p,ensure_ascii=False,indent=2),encoding='utf-8')

# ComfyUIサーバー側で検証される。構成が変わっていたら詳細エラーを表示する。
start=time.perf_counter()
resp=requests.post(URL+'/prompt',json={'prompt':p,'client_id':str(uuid.uuid4())},timeout=120)
assert resp.ok, f'ComfyUIのワークフロー検証失敗: {resp.status_code} {resp.text[:4000]}'
info=resp.json()
assert 'prompt_id' in info, 'ワークフロー拒否: '+str(info)[:3000]
pid=info['prompt_id']; print('✅ キュー登録:',pid,flush=True)
STATUS='running'; history_entry=None
try:
    while True:
        time.sleep(10)
        hist=requests.get(URL+'/history/'+pid,timeout=60).json()
        if pid in hist:
            history_entry=hist[pid]
            stat=history_entry.get('status',{})
            STATUS=str(stat.get('status_str','completed'))
            if STATUS!='success':
                raise RuntimeError('推論失敗：'+json.dumps(stat.get('messages',[]),ensure_ascii=False)[-5000:])
            break
        elapsed=time.perf_counter()-start
        q=requests.get(URL+'/queue',timeout=30).json()
        running_ids=[str(x[1]) for x in q.get('queue_running',[]) if len(x)>1]
        queued_ids=[str(x[1]) for x in q.get('queue_pending',[]) if len(x)>1]
        state='生成中' if pid in running_ids else '順番待ち' if pid in queued_ids else '処理中'
        print(f'⏳ {state} — 経過 {elapsed/60:.1f}分',flush=True)
except Exception:
    STATUS='failed'
    raise
except KeyboardInterrupt:
    STATUS='interrupted'
    print('⚠ このセルの停止だけではサーバー側のジョブが続く場合があります。',flush=True)
    raise
finally:
    elapsed=time.perf_counter()-start
    RUN_TIMING={'status':STATUS, 'elapsed_sec':round(elapsed,2),'gpu':cmd_out(['nvidia-smi','--query-gpu=name','--format=csv,noheader']),
      'source_audio_sec':audio_duration,'reference_audio_sec':ref_seconds,'video_frames':FRAMES,'full_audio_intended_sec':round(requested_sec,3),
      'target_video_sec':CLIP_SEC,'resolution':f'{WIDTH}x{HEIGHT}', 'steps':NUM_STEPS,
      'turbo_4_step':TURBO_4_STEP, 'step_mode':STEP_MODE, 'lora_name':LORA_NAME,'prompt_id':pid,'job':JOB.name,'prompt':MOTION_PROMPT}
    (JOB/'generation_time.json').write_text(json.dumps(RUN_TIMING,ensure_ascii=False,indent=2),encoding='utf-8')
print('✅ 動画生成処理完了！所要時間:',f'{elapsed/60:.2f}分')

# ComfyUIのhistoryが返す出力を明示的に探す（古い動画を誤取得しない）
vids=[]
for node in history_entry.get('outputs',{}).values():
    for v in node.get('videos',[]):
        fname=v.get('filename')
        if not fname: continue
        path=(COMFY/'output'/v.get('subfolder','')/fname).resolve()
        if path.is_file() and path.is_relative_to((COMFY/'output').resolve()): vids.append(path)
if not vids:
    # ComfyUIバージョンによって動画キーが変化した場合もprefixで検索
    vids=sorted((COMFY/'output').rglob('H3_Ruka_'+JOB.name+'*.mp4'),key=lambda x:x.stat().st_mtime,reverse=True)
assert vids, '動画ファイルが見つかりません。ComfyUI/outputと履歴を確認してください。'
H3_VIDEO=JOB/'MiniMax_H3_generated.mp4'
shutil.copy2(vids[0],H3_VIDEO)
print('✅ 元の生成MP4:',H3_VIDEO,'サイズ',round(H3_VIDEO.stat().st_size/2**20,1),'MiB')

In [ ]:
#@title ⑦ 元音声比較版・時間レポート・ZIPを作成してダウンロード
import json, subprocess, shutil, zipfile, time
from pathlib import Path
from IPython.display import display, HTML, FileLink
from google.colab import files
assert H3_VIDEO.exists(),'先に⑥で動画を生成してください'
FINAL_ORIGINAL=JOB/'MiniMax_H3_with_original_audio.mp4'
finalize_start=time.perf_counter()
# v5: ⑥で作る reference_audio.wav は映像生成用に短くなる場合がある。
# 音声差替えには元の AUDIO_PATH を用いる（参照音声を使うと末尾0.625秒などが消える）。
FINAL_DURATION = min(audio_duration, TEST_SECONDS) if TEST_SECONDS > 0 else audio_duration
VIDEO_DEFICIT = max(0.0, FINAL_DURATION - CLIP_SEC)
ffmpeg_base=['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(H3_VIDEO),'-i',str(AUDIO_PATH),'-map','0:v:0','-map','1:a:0']
if VIDEO_DEFICIT > 0.001:
    # 映像の追加生成はしない。最後の画像を不足分だけ延長する。
    video_opts=['-vf',f'tpad=stop_mode=clone:stop_duration={VIDEO_DEFICIT+0.08:.4f}',
                '-c:v','libx264','-preset','veryfast','-crf','18','-pix_fmt','yuv420p']
else:
    video_opts=['-c:v','copy']
cmd=ffmpeg_base+video_opts+['-c:a','aac','-b:a','192k','-af','apad',
                            '-t',str(FINAL_DURATION),'-movflags','+faststart',str(FINAL_ORIGINAL)]
subprocess.run(cmd,check=True)
# ファイルの実際の再生尺も取得し、ユーザーに表示する。
probed=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration',
    '-of','json',str(FINAL_ORIGINAL)],text=True))
ACTUAL_FINAL_DURATION=float(probed['format']['duration'])
print(f'✅ 元音声比較版: {ACTUAL_FINAL_DURATION:.3f}秒（元音声の使用尺 {FINAL_DURATION:.3f}秒）')
if VIDEO_DEFICIT > 0.001:
    print(f'⚠ 末尾約{VIDEO_DEFICIT:.3f}秒は最後の映像フレームを静止延長しています。口パクは追加生成されません。')
RUN_TIMING['finalize_sec']=round(time.perf_counter()-finalize_start,2)
RUN_TIMING['final_mp4_sec']=round(ACTUAL_FINAL_DURATION,3)
RUN_TIMING['final_original_audio_sec']=round(FINAL_DURATION,3)
RUN_TIMING['freeze_frame_extension_sec']=round(VIDEO_DEFICIT,3)
RUN_TIMING['note']='Generated MP4 uses H3 audio. Original-audio version uses full original narration and may freeze the last frame to avoid truncation. Lip sync is not corrected.'
(JOB/'generation_time.json').write_text(json.dumps(RUN_TIMING,ensure_ascii=False,indent=2),encoding='utf-8')
report=f"""MiniMax H3 / Ref2VA 生成結果
------------------------
状態: {RUN_TIMING['status']}
GPU: {RUN_TIMING['gpu']}
H3生成映像の長さ: {CLIP_SEC:.2f}秒\n元音声比較版の長さ: {ACTUAL_FINAL_DURATION:.2f}秒\n静止延長した長さ: {VIDEO_DEFICIT:.2f}秒
元音声の長さ: {audio_duration:.2f}秒
生成時の参照音声: {ref_seconds:.2f}秒\n差し替え元音声の使用尺: {FINAL_DURATION:.2f}秒
動画生成時間: {RUN_TIMING['elapsed_sec']/60:.2f}分 ({RUN_TIMING['elapsed_sec']:.1f}秒)
音声差替え時間: {RUN_TIMING['finalize_sec']:.1f}秒
モード: Ref2VA / {str(NUM_STEPS)+'-step Turbo' if USE_TURBO else '20-step Base'}
解像度: {WIDTH} x {HEIGHT}
フレーム数: {FRAMES}
注意: 参照音声のセリフ・タイミングが忠実に再現される保証はありません。
"""
(JOB/'generation_time.txt').write_text(report,encoding='utf-8')
print(report)
display(HTML('<h3>✅ 生成完了：結果表</h3><table border="1" cellpadding="6">'+''.join(
  f'<tr><td>{a}</td><td>{b}</td></tr>' for a,b in [
   ('GPU',RUN_TIMING['gpu']),('H3動画尺',f'{CLIP_SEC:.2f}秒'),('元音声版動画尺',f'{ACTUAL_FINAL_DURATION:.2f}秒'),('生成時間',f'{RUN_TIMING["elapsed_sec"]/60:.2f}分'),
   ('手法',f'Turbo {NUM_STEPS} steps' if USE_TURBO else 'Base 20 steps'),('解像度',f'{WIDTH}×{HEIGHT}')])+'</table>'))
try:
    from IPython.display import Video
    display(Video(str(H3_VIDEO),embed=True,width=560))
except Exception as e:
    print('プレビューのみ失敗（動画は保存済み）:',str(e)[:200])
ZIP=JOB/'MiniMax_H3_video_and_timing.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for f in [H3_VIDEO,FINAL_ORIGINAL,JOB/'generation_time.txt',JOB/'generation_time.json',JOB/'workflow_api.json']:
        z.write(f,f.name)
print('✅ ZIP:',ZIP,'\n内容: 生成音声MP4、元音声比較MP4、時間レポート、実行設定')
try:
    files.download(str(ZIP))
except Exception as e:
    print('自動ダウンロードに失敗。左のファイル一覧から取得してください:',ZIP,e)

## メモ・トラブル対処

- **音声が自分のセリフと一致しない:** これは必ずしもエラーではありません。Ref2VAは音声を**参照**に使いますが、既存の音声を完全に複製する保証はありません。ZIP内の `MiniMax_H3_with_original_audio.mp4` と比較してください。
- **メモリ不足:** A100 40GBで高解像度が厳しい場合は②を「横長 832x480」、テスト5秒のままにし、⑤のComfyUI起動引数に `--lowvram` を付けて起動し直してください。システムRAMにも十分な空きが必要です。
- **モデル取得に失敗:** ④のみ再実行。Hugging Face側の一時的な混雑やディスク容量も確認してください。
- **キューが終わらない:** ⑤で表示した `comfyui.log` の末尾を確認。生成セルを止めてもバックグラウンドのComfyUIが続行する場合があります。
- **生成済みの出力:** `/content/h3_ruka_test/outputs/` に残ります（Colabランタイム削除時には消えます）。
- **40秒や10分の動画:** H3 Ref2VAは短いクリップで評価し、最終的に多数のカットと実演画面を編集でつなぐ方針。長尺を1回で生成する用途ではありません。
- **モデル利用条件:** 商用動画制作前に [MiniMax H3コミュニティライセンス](https://huggingface.co/MiniMaxAI/MiniMax-H3) を確認してください。

**GPUコスト:** モデルのダウンロードや環境構築中もColabのGPU接続時間はCUの対象になります。利用枠が少ない場合は注意してください。
- **8ステップ切替手順:** ②で「8ステップ」を選ぶ→④でLoRA取得→⑤でノード確認→⑥生成→⑦動画とレポート。①のアップロードは入力変更がないなら再実行不要。すでに⑤が起動している場合、LoRA追加後も⑥から試せます。
- **注意:** 8ステップLoRAは768p学習モデル。既存832x480設定でも技術的には実行できますが、最適画質は保証しません。A100 40GBはVRAMが非常に逼迫するので、まず5秒で確認してください。

- **v4:** 初期設定で音声を自動使用します。15秒の音声でも生成モデル側のフレーム制限により最大約14.38秒です。②で `TEST_SECONDS=5` とすると意図的に先頭5秒だけになります。8ステップを選ぶ際は④で専用LoRAを取得してください。

- **v5:** 15秒の音声に対して生成映像が14.375秒の場合も、元音声比較版では最後の映像を静止延長して15秒の元音声を残します。これは末尾の口パクを補完するものではありません。H3生成音声版は元モデルの出力尺（最大約14.38秒）のままです。